<a href="https://colab.research.google.com/github/tmulderrig/vida_yellowstone_finemapping/blob/main/zero_shot_vida_yellowstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Basic examples with PlantCaduceus

### Setup environment

In [ ]:
!nvcc --version

## Install specific pytorch
We are installing a specific version of PyTorch (2.3.1) to resolve some errors when loading mamba-ssm, The most critical part is `--index-url https://download.pytorch.org/whl/cu121`. This command tells pip to NOT use the default repository. Instead, it downloads a special version of PyTorch that was pre-compiled specifically for systems with a CUDA 12.1 driver.

#### Why is this necessary?
Google Colab has its own system-level NVIDIA driver (e.g., CUDA 12.5). By installing a PyTorch build that is aware of a compatible CUDA version (12.1 works perfectly with a 12.5 driver), we ensure that all parts of the library, including low-level components like Triton, can find the correct drivers and compile code successfully during model inference. This fixes both the `libcuda.so` and the `ModuleNotFoundError` errors.

In [ ]:
!pip3 install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121

In [ ]:
!pip3 install mamba-ssm==2.2.2 transformers==4.40.0 git+https://github.com/dridk/PyVCF3.git@master scipy==1.12.0 biopython xgboost==2.0.3 scikit-learn==1.4.0 matplotlib

🔄 You may need to restart the session after running the above setup cells.

Once restarted, you’re all set to start exploring the PlantCAD model!

## Testing if mamba-ssm is installed successfully

In [ ]:
# Test core dependencies
import torch
from mamba_ssm import Mamba
from transformers import AutoTokenizer, AutoModelForMaskedLM
import pandas as pd

device = 'cuda:0'

# Test PlantCAD model loading
tokenizer = AutoTokenizer.from_pretrained('kuleshov-group/PlantCaduceus_l32')
model = AutoModelForMaskedLM.from_pretrained('kuleshov-group/PlantCaduceus_l32', trust_remote_code=True)
model.to(device)
print("✅ Installation successful!")

## Run some examples on zero-shot mutation effect prediction

In [ ]:
# clone PlantCAD repo
!git clone https://github.com/kuleshov-group/PlantCaduceus.git

In [ ]:
!wget https://urgi.versailles.inrae.fr/download/iwgsc/IWGSC_RefSeq_Assemblies/v1.0/iwgsc_refseqv1.0_chr2D.fsa.zip
!unzip iwgsc_refseqv1.0_chr2D.fsa.zip

In [ ]:
# Upload VCF
from google.colab import files
uploaded = files.upload()
uploaded_vcf = list(uploaded.keys())[0]


# Decompress if needed
!gunzip -c "{uploaded_vcf}" > input.vcf || cp "{uploaded_vcf}" input.vcf

In [ ]:
# Download wheat chromosome 4A reference sequence
!wget https://urgi.versailles.inrae.fr/download/iwgsc/IWGSC_RefSeq_Assemblies/v1.0/iwgsc_refseqv1.0_chr4A.fsa.zip

!unzip -o iwgsc_refseqv1.0_chr4A.fsa.zip


In [ ]:
#Run PlantCaduceus
!python PlantCaduceus/src/zero_shot_score.py \
    -input-vcf input.vcf \
    -input-fasta iwgsc_refseqv1.0_chr4A.fsa \
    -output zero_shot_4A.vcf \
    -model 'kuleshov-group/PlantCaduceus_l32' \
    -device 'cuda:0'

In [ ]:
files.download('zero_shot_4A.vcf')

In [ ]:
!grep -v '#' zero_shot_4A.vcf | awk  -v OFS='\t' '{print $1,$2,$4,$5,$8}' | head -n 20